# Instalacao das bibliotecas


In [22]:
!pip install numpy==1.23.5 pandas==2.2.2

In [23]:
!pip install pmdarima

In [24]:
!pip install deap

# Importacao das bibliotecas e dos dados


In [25]:
import pandas as pd
import os
import numpy as np

In [26]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from deap import base, creator, tools, algorithms
import random

In [27]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
from pmdarima import auto_arima
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import joblib

In [28]:

from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [29]:
caminho_dados = '/content/drive/Shareddrives/IC - Otimizacao computacional para inflacao/Dados/dados_USA'

# Listar os arquivos dentro da pasta "Dados"
arquivos_dados = os.listdir(caminho_dados)

print(arquivos_dados)


['Untitled0.ipynb', 'dados_macroeconomicos_usa.csv']


In [30]:
# Caminho para o arquivo CSV
caminho_arquivo = os.path.join(caminho_dados, 'dados_macroeconomicos_usa.csv')

# Ler o arquivo CSV
df = pd.read_csv(caminho_arquivo)

In [31]:
df = df.set_index('DATE')
df.drop(columns=['Unnamed: 0'], inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 180 entries, 2010-01-01 to 2024-12-01
Data columns (total 19 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Inflacao_CPI                 180 non-null    float64
 1   Inflacao_CPI_Core            180 non-null    float64
 2   PCE_Core                     180 non-null    float64
 3   Desemprego                   180 non-null    float64
 4   Desemprego_menos_27_semanas  180 non-null    float64
 5   Taxa_Juros_Fed               180 non-null    float64
 6   Emprego_Total_Nao_Agricola   180 non-null    int64  
 7   Permissoes_Construcao        180 non-null    int64  
 8   Salario_Medio                180 non-null    float64
 9   Salario_Medio_Real_Hora      180 non-null    float64
 10  Producao_Industrial          180 non-null    float64
 11  Capacidade_Instalada         180 non-null    float64
 12  PCE                          180 non-null    float64
 13  Deflator_

# Divisao dos dados e treinamento

In [32]:
# Divisao do alvo e dados de treino
X = df.drop(columns = ['Inflacao_CPI'])
y = df['Inflacao_CPI']

In [33]:
# Normalizacao dos dados usando MinMax
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)
X = pd.DataFrame(X_scaled, columns=X.columns)

# Split dos dados com janela de 5
tscv = TimeSeriesSplit(n_splits=5)

# Criacao da funcao fitness

In [34]:
def feature_selection_fitness(individual, model, X_scaled, y, tscv, metric_func, max_features_allowed=None):
    """
    Parâmetros:
    - individual: cromossomo binário representando as features selecionadas
    - model: modelo de aprendizado
    - X_scaled: dados de entrada escalados (pandas DataFrame)
    - y: variável alvo (pandas Series)
    - tscv: TimeSeriesSplit
    - metric_func: função para calcular a métrica
    - max_features_allowed: número máximo de features permitidas (int ou None)

    Retorna:
    - métrica de avaliação para o conjunto de features selecionadas
    """

    # Garantir que X_scaled seja um DataFrame
    if isinstance(X_scaled, np.ndarray):
        X_scaled = pd.DataFrame(X_scaled, columns=[f"feature_{i}" for i in range(X_scaled.shape[1])])

    selected_features = [feature for i, feature in enumerate(X_scaled.columns) if individual[i] == 1]

    # Penalizar se não houver features ou se exceder o máximo permitido
    if len(selected_features) == 0:
        return -9999.,
    if max_features_allowed is not None and len(selected_features) > max_features_allowed:
        return -9999.,

    # Realiza a divisão temporal dos dados
    train_index, test_index = list(tscv.split(X_scaled))[-1]

    X_train, X_test = X_scaled.iloc[train_index], X_scaled.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    try:
        model.fit(X_train[selected_features], y_train)
        y_pred = model.predict(X_test[selected_features])
        score = metric_func(y_test, y_pred)
    except Exception as e:
        print(f"Erro ao treinar ou avaliar o modelo: {e}")
        return -9999.,

    return score,


# Funcoes auxiliares

In [35]:
#Funcoes auxiliares

def make_stationary(df, max_diff=5):
    """Torna a série temporal estacionária através de diferenciação."""
    df_diff = df.copy()
    for i in range(1, max_diff + 1):
        stationary = True
        for column in df_diff.columns:
            if not check_stationarity(df_diff[[column]]):
                stationary = False
                df_diff[[column]] = df_diff[[column]].diff().fillna(method='ffill').fillna(method='bfill')
        if stationary:
            print(f"Série estacionária após {i-1} diferenciações.")
            return df_diff.dropna()
    raise ValueError("A série não se tornou estacionária após o número máximo de diferenciações.")

def check_stationarity(series, significance_level=0.05):
    """Verifica se a série é estacionária usando o teste ADF."""
    from statsmodels.tsa.stattools import adfuller
    result = adfuller(series.dropna())
    return result[1] < significance_level

def plot_forecasts(y_test, final_forecast, title='Previsão Dinâmica'):
    """Plota as previsões comparadas com os valores reais."""
    plt.figure(figsize=(15, 3))
    plt.plot(y_test.index, y_test, label='Valores Reais', color='blue')
    plt.plot(y_test.index, final_forecast, label='Previsão', color='orange')
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()

# ARIMAX MODEL

In [36]:
def executar_modelo_arimax(df, exogenous_vars):

    # Tornar o DataFrame estacionário
    df_stationary = make_stationary(df[['Inflacao_CPI'] + exogenous_vars])

    # Dividir em treino e teste
    train_size = int(len(df_stationary) * 0.8)
    train, test = df_stationary.iloc[:train_size], df_stationary.iloc[train_size:]

    # Variável dependente e exógenas
    y_train = train['Inflacao_CPI']
    y_test = test['Inflacao_CPI']
    X_train = train[exogenous_vars]
    X_test = test[exogenous_vars]

    # Ajustar modelo ARIMAX com auto_arima
    arimax_model = auto_arima(y_train, exogenous=X_train, seasonal=False, trace=True,
                               error_action='ignore', suppress_warnings=True)
    arimax_order = arimax_model.order

    # Previsão dinâmica com atualização a cada passo
    dynamic_arimax_forecast = []
    history_y = list(y_train)
    history_X = list(X_train.values)

    for t in range(len(y_test)):
        # Ajusta SARIMAX com histórico até o ponto atual
        model = SARIMAX(history_y, exog=np.array(history_X), order=arimax_order)
        fitted_model = model.fit(disp=False)

        # Previsão de 1 passo à frente
        next_x = X_test.iloc[t].values.reshape(1, -1)
        forecast = fitted_model.forecast(steps=1, exog=next_x)
        dynamic_arimax_forecast.append(forecast[0])

        # Atualiza histórico com valores reais
        history_y.append(y_test.iloc[t])
        history_X.append(X_test.iloc[t].values)

    # Converter previsões para DataFrame
    final_forecast_df = pd.DataFrame(dynamic_arimax_forecast, index=y_test.index, columns=['Final_Forecast'])
    final_forecast_df.to_csv('arimax_pred.csv')

    # Salvar o modelo original do auto_arima (pode ser útil para reuso)
    joblib.dump(arimax_model, 'arimax_model.pkl')

    # Cálculo dos resíduos
    residuals = y_test.values - dynamic_arimax_forecast
    residuals_df = pd.DataFrame(residuals, index=y_test.index, columns=['Residuals'])
    residuals_df.to_csv('arimax_res.csv')

    # Métricas de desempenho
    mse = mean_squared_error(y_test, dynamic_arimax_forecast)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, dynamic_arimax_forecast)
    mape = np.mean(np.abs((y_test - dynamic_arimax_forecast) / y_test))

    #print(f'MAE: {mae:.6f}')
    #print(f'MSE: {mse:.6f}')
    #print(f'RMSE: {rmse:.6f}')
    #print(f'MAPE: {mape:.6f}')


    return mae

    # Plotar resultados
    #plot_forecasts(y_test, dynamic_arimax_forecast, title=tecnica_nome)

# Funcao fitness arimax

In [37]:
def arimax_feature_fitness(individual, df, all_features):
    """
    Calcula o fitness do indivíduo treinando e avaliando um modelo ARIMAX
    """
    import statsmodels.api as sm
    from sklearn.metrics import mean_squared_error

    # Seleciona as features ativas no indivíduo
    selected_features = [feature for gene, feature in zip(individual, all_features) if gene == 1]

    if len(selected_features) == 0:
        return (9999.,)  # Penalidade para indivíduos sem features

    # Preparar dados
    y = df['Inflacao_CPI']
    X = df[selected_features]

    # Separar treino/teste (aqui exemplo com 80/20%)
    split_idx = int(len(df) * 0.8)
    y_train, y_test = y[:split_idx], y[split_idx:]
    X_train, X_test = X[:split_idx], X[split_idx:]

    try:
        # Treina o modelo ARIMAX
        model = sm.tsa.SARIMAX(
            endog=y_train,
            exog=X_train,
            order=(1, 0, 0),  # Ajuste conforme necessário
            seasonal_order=(0, 0, 0, 0),
            enforce_stationarity=False,
            enforce_invertibility=False
        )
        results = model.fit(disp=False)

        # Faz predição
        y_pred = results.predict(start=split_idx, end=len(df)-1, exog=X_test)

        # Calcula a métrica de erro
        mae = mean_absolute_error(y_test, y_pred)

        # === Imprimir informações desejadas ===
        print("\nAvaliação do Modelo Atual:")
        print("Features Exógenas Utilizadas:", selected_features)
        print("MSE (Erro Quadrático Médio):", mae)

    except Exception as e:
        print(f"Erro no ajuste ou na predição do ARIMAX: {e}")
        return (9999.,)  # Penaliza fortemente caso falhe

    return (mae,)


# Implementacao da selecao

In [38]:
import warnings
warnings.filterwarnings('ignore')

In [45]:
from deap import base, creator, tools, algorithms
import random
import numpy as np
import os

# Semente para reprodutibilidade
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# Lista de todas as variáveis exógenas disponíveis
ALL_FEATURES = df.columns.drop('Inflacao_CPI').tolist()

# Definindo o número máximo de features permitidas
MAX_FEATURES_ALLOWED = 5

# Criando as classes de fitness e indivíduo
creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.FitnessMin)

toolbox = base.Toolbox()

# Função para criar um indivíduo respeitando o limite de features
def create_individual():
    individual = [0] * len(ALL_FEATURES)
    selected_indices = random.sample(range(len(ALL_FEATURES)), MAX_FEATURES_ALLOWED)
    for idx in selected_indices:
        individual[idx] = 1
    return creator.Individual(individual)

toolbox.register("individual", create_individual)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

# Função de avaliação (fitness) usando ARIMAX
toolbox.register("evaluate", lambda ind: arimax_feature_fitness(ind, df, ALL_FEATURES))

# Cruzamento
toolbox.register("mate", tools.cxTwoPoint)

# Função de mutação que respeita o limite de features
def mutate(individual, indpb):
    for i in range(len(individual)):
        if random.random() < indpb:
            if individual[i] == 1:
                individual[i] = 0
            else:
                if sum(individual) < MAX_FEATURES_ALLOWED:
                    individual[i] = 1
    if sum(individual) == 0:
        idx = random.randrange(len(individual))
        individual[idx] = 1
    return individual,

toolbox.register("mutate", mutate, indpb=0.2)

# Seleção
toolbox.register("select", tools.selTournament, tournsize=3)

# ===== Função auxiliar para garantir o limite de features após cruzamento =====
def limit_features(individual):
    active_indices = [i for i, gene in enumerate(individual) if gene == 1]
    if len(active_indices) > MAX_FEATURES_ALLOWED:
        to_deactivate = random.sample(active_indices, len(active_indices) - MAX_FEATURES_ALLOWED)
        for idx in to_deactivate:
            individual[idx] = 0
    return individual

In [46]:
# Parâmetros do GA
population_size = 8
generations = 10
crossover_prob = 0.7
mutation_prob = 0.2

# População inicial
population = toolbox.population(n=population_size)

# ====== Loop do GA manual para controlar impressões ======

for gen in range(generations):
    print(f"\n--- Geração {gen+1} ---")

    # Cruzamento
    offspring = algorithms.varAnd(population, toolbox, cxpb=crossover_prob, mutpb=mutation_prob)

    # Avaliar indivíduos que ainda não possuem fitness calculado
    invalid_ind = [ind for ind in offspring if not ind.fitness.valid]

    for ind in invalid_ind:
        # Avalia e já imprime os exógenos usados
        fitness = toolbox.evaluate(ind)
        ind.fitness.values = fitness

    # Substituir população
    population[:] = toolbox.select(offspring, k=len(population))

    # Melhor indivíduo da geração
    best_ind = tools.selBest(population, k=1)[0]
    selected_features = [f for gene, f in zip(best_ind, ALL_FEATURES) if gene == 1]

    print("Melhores features nesta geração:", selected_features)
    print("Fitness:", best_ind.fitness.values[0])

# ====== Fim do GA ======

# Melhor solução final
best_ind = tools.selBest(population, k=1)[0]
selected_features = [f for gene, f in zip(best_ind, ALL_FEATURES) if gene == 1]

print("\n=== Melhor conjunto de features encontrado ===")
print("Features:", selected_features)
print("Fitness final:", best_ind.fitness.values[0])



--- Geração 1 ---

Avaliação do Modelo Atual:
Features Exógenas Utilizadas: ['Inflacao_CPI_Core', 'Desemprego_menos_27_semanas', 'Salario_Medio_Real_Hora', 'Producao_Industrial', 'PIB', 'TIPS_5_anos_infl_ajustado']
MSE (Erro Quadrático Médio): 13.370566954857116

Avaliação do Modelo Atual:
Features Exógenas Utilizadas: ['Desemprego', 'Desemprego_menos_27_semanas', 'Taxa_Juros_Fed', 'Permissoes_Construcao']
MSE (Erro Quadrático Médio): 46.334171078278445

Avaliação do Modelo Atual:
Features Exógenas Utilizadas: ['Inflacao_CPI_Core', 'PCE_Core', 'Desemprego', 'Desemprego_menos_27_semanas', 'PIB']
MSE (Erro Quadrático Médio): 7.176288727346468

Avaliação do Modelo Atual:
Features Exógenas Utilizadas: ['Inflacao_CPI_Core', 'Permissoes_Construcao', 'Capacidade_Instalada', 'PCE', 'TIPS_5_anos_nominal']
MSE (Erro Quadrático Médio): 3.958095583077906

Avaliação do Modelo Atual:
Features Exógenas Utilizadas: ['Salario_Medio', 'Producao_Industrial', 'PIB_Real', 'Indice_Dolar', 'TIPS_5_anos_infl